# Script 8 - Rhythm metrics to outcomes, total / direct effects (Supplementary Table S15)

Estimates the standardised rhythm -> PHQ-9 depression and rhythm -> cognition
composite associations for each of intradaily variability (IV), interdaily
stability (IS) and relative amplitude (RA), using survey-weighted multivariable
linear regression (WLS) with HC3 heteroscedasticity-consistent standard errors.
Three specifications per outcome:

- M1 (total effect, $c$-path): covariates age, sex, body mass index, education,
  accelerometer wear time, self-reported sleep duration; no mediator in the model.
- M2 (+ total volume): M1 + total daily activity volume (TOTAL_MIMS). Sensitivity.
- M3 (direct effect, $c'$-path): M1 + both biological mediators (KDM-BA
  advancement, log SII). The exposure coefficient is the direct effect
  conditional on the mediators, the analogue of the Mplus $c'$ path (Table 2).

Depression is analysed in adults aged 18-59, 60+ and 18+; cognition is measured
only at 60+ in NHANES.

## Total / direct decomposition

$$c = c' + a_1 b_1 + a_2 b_2$$

where $c$ = total (M1), $c'$ = direct (M3), and $a_j b_j$ = indirect effect via
mediator $j$ (from the SEM, Table 2).

## Per-outcome samples and survey-weighted standardisation

Each regression uses a per-outcome sample (exposure, outcome, weight and all
required predictors non-null) and z-standardises the exposure, the outcome and
the mediators using the survey-weighted mean and SD on that sample (Mplus STDYX
convention under TYPE = COMPLEX + MLR):

$$\hat{\mu}_w = \frac{\sum_i w_i x_i}{\sum_i w_i}, \quad \hat{\sigma}_w = \sqrt{\frac{\sum_i w_i (x_i - \hat{\mu}_w)^2}{\sum_i w_i}}, \quad z_i = \frac{x_i - \hat{\mu}_w}{\hat{\sigma}_w}$$

## Inputs

- `data/processed/8_nhanes_analytic.csv`

## Outputs

- `result/supplementary/SuppTableS15_TotalDirect_WLS.csv` and `.xlsx`
- `result/raw/script8_outcomes_full.csv`


## Step 0 — Imports and paths


In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.options.display.max_columns = 60

import os as _os

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = _os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE = find_root()
PROC   = BASE / 'data' / 'processed'
R_MAIN = BASE / 'result' / 'main'
R_SUPP = BASE / 'result' / 'supplementary'
R_RAW  = BASE / 'result' / 'raw'
for p in (R_MAIN, R_SUPP, R_RAW):
    p.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook uses the code on disk.
import importlib
import wls_utils, nature_palette
for _m in (nature_palette, wls_utils,):
    importlib.reload(_m)

from wls_utils import wls_fit

## Step 1 — Load 8_nhanes_analytic.csv, rename columns, derive PHQ-9 and the cognition composite


In [ ]:
df = pd.read_csv(PROC / '8_nhanes_analytic.csv')
print(f'Loaded 8_nhanes_analytic.csv: {df.shape}')

rename = {
    'RIDAGEYR':       'AGE',
    'RIAGENDR':       'SEX',
    'BMXBMI':         'BMI',
    'DMDEDUC2':       'EDU',
    'Wear_Time':      'WEAR_T',
    'SLD010H':        'SLEEP_H',
    'obj_mvpa_min':   'OBJ_MVPA',
    'SDMVSTRA':       'STRATUM',
    'SDMVPSU':        'PSU',
    'KDM_BA_advance': 'KDM_BA_ADV',
    'SII_log':        'SII_LOG',
}
df = df.rename(columns=rename)
# WTMEC4YR is stored in the analytic file; checked here, not re-derived.
assert 'WTMEC4YR' in df.columns, 'WTMEC4YR missing from the analytic file'
assert np.allclose(df['WTMEC4YR'], df['WTMEC2YR'] / 2.0, equal_nan=True), \
    'stored WTMEC4YR does not match WTMEC2YR / 2 -- the file is stale'
df = df[df['AGE'] >= 18].copy()
print(f'Adults 18+: n = {len(df):,}')

# PHQ-9 total: all nine items required (min_count = 9)
dpq_cols = [f'DPQ{i:03d}' for i in [10, 20, 30, 40, 50, 60, 70, 80, 90]]
phq = df[dpq_cols].replace({7: np.nan, 9: np.nan})
df['PHQ9'] = phq.sum(axis=1, min_count=9)

# Cognition composite z: raw 0 -> NaN, z within the 60+ subsample, all three domain z-scores required
elig_60 = df['AGE'] >= 60
for src_col, out_col in [('memory_score', 'MEM_Z'),
                          ('fluency_score', 'FLUEN_Z'),
                          ('speed_score',  'SPEED_Z')]:
    raw = df[src_col].replace(0, np.nan)
    s = raw.loc[elig_60].dropna()
    mu = s.mean(); sd = s.std()
    df[out_col] = np.where(elig_60, (raw - mu) / sd, np.nan)
df['COG_Z'] = df[['MEM_Z', 'FLUEN_Z', 'SPEED_Z']].mean(axis=1, skipna=False)

print(f'  PHQ-9 non-null: {df["PHQ9"].notna().sum():,}')
print(f'  COG_Z non-null (60+ with all 3 indicators): {df["COG_Z"].notna().sum():,}')

## Step 2 — Helpers: per-outcome sample and survey-weighted mean / SD

M1 covariates: age, sex, BMI, education, wear time, sleep duration. M2 adds
TOTAL_MIMS; M3 adds KDM_BA_ADV and SII_LOG.


In [ ]:
M1_COVS    = ['AGE', 'SEX', 'BMI', 'EDU', 'WEAR_T', 'SLEEP_H']
# Total daily activity volume (TOTAL_MIMS): correlates 0.903 with threshold-MVPA
# and needs no cut-point (see script 7).
M2_EXTRA   = ['TOTAL_MIMS']

# Depression age cells: 18-59, 60+ and pooled 18+. Cognition is measured only
# at 60+ in NHANES.
AGE_CELLS = [('18-59', 18, 60), ('60+', 60, 999), ('18+', 18, 999)]
MED_VARS   = ['KDM_BA_ADV', 'SII_LOG']

def per_outcome_sample(exposure, outcome_col, extra_required=(), age_min=18, age_max=999):
    needed = [exposure, outcome_col, 'WTMEC4YR'] + M1_COVS + list(extra_required)
    mask = ((df['AGE'] >= age_min) & (df['AGE'] < age_max)
            & df[needed].notna().all(axis=1))
    return df.loc[mask].copy()

def weighted_mean_sd(x, w):
    w_tot = np.sum(w)
    mu = np.sum(x * w) / w_tot
    var = np.sum(w * (x - mu) ** 2) / w_tot
    return mu, float(np.sqrt(var))

from nature_palette import EXP_ORDER
from nature_palette import fmt_ci as _fmt_ci, age_label
EXPOSURES = list(EXP_ORDER)   # RA, IS, IV
EXP_NAME  = {'IV': 'Intradaily variability', 'IS': 'Interdaily stability',
             'RA': 'Relative amplitude'}


print('Per-outcome sample sizes (IV; IS matches, RA differs by <= 8):')
for outcome_col, lab, cells in [('PHQ9', 'Depression (PHQ-9)', AGE_CELLS),
                                ('COG_Z', 'Cognition (60+)', [('60+', 60, 999)])]:
    print(f'  {lab}')
    for cell, lo, hi in cells:
        ns = [len(per_outcome_sample('IV', outcome_col, e, lo, hi))
              for e in ([], M2_EXTRA, MED_VARS)]
        print(f'    {cell:6s}  M1 {ns[0]:6,}   M2 (+volume) {ns[1]:6,}   M3 (+mediators) {ns[2]:6,}')


## Step 3 — WLS-HC3 with sample-specific survey-weighted standardisation

For one outcome and one specification:

1. Filter to the per-outcome sample (exposure, outcome, weight, covariates and extras non-null).
2. Compute the survey-weighted mean and SD of the exposure, the outcome and the mediators (when present) on that sample.
3. Z-standardise the exposure, the outcome and the mediators with those weighted SDs.
4. Fit survey-weighted WLS with HC3 SE on the standardised values.
5. The exposure coefficient is the STDYX beta, comparable to the Mplus $c$-path (M1) or $c'$-path (M3).

Covariates AGE, SEX, BMI, EDU, WEAR_T, SLEEP_H and TOTAL_MIMS stay on their raw
scales (Mplus STDYX convention).


In [ ]:
def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def fmt_ci(est, lo, hi, dp=3):
    return _fmt_ci(est, lo, hi, dp)   # minus kept, plus omitted

def run_y_path_stdyx(exposure, outcome_col, extras, age_min=18, age_max=999):
    """WLS-HC3 of the outcome on one rhythm metric + covariates (+ extras), z-standardised with survey-weighted SDs on the per-outcome sample."""
    ana = per_outcome_sample(exposure, outcome_col, extras, age_min, age_max)
    w   = ana['WTMEC4YR'].values

    mu_iv,  sd_iv  = weighted_mean_sd(ana[exposure].values,     w)
    mu_out, sd_out = weighted_mean_sd(ana[outcome_col].values,  w)
    iv_z   = (ana[exposure].values     - mu_iv)  / sd_iv
    out_z  = (ana[outcome_col].values  - mu_out) / sd_out

    cols = [iv_z]
    for extra in extras:
        if extra in MED_VARS:
            mu_m, sd_m = weighted_mean_sd(ana[extra].values, w)
            cols.append((ana[extra].values - mu_m) / sd_m)
        else:
            cols.append(ana[extra].values)  # volume kept on its raw scale
    for c in M1_COVS:
        cols.append(ana[c].values)
    X = np.column_stack(cols)
    fit = wls_fit(X, out_z, w=w, se='hc3', add_const=True)

    return {
        'exposure':    exposure,
        'outcome':     outcome_col,
        'age_cell':    f'{age_min}-{age_max}',
        'extras':      ', '.join(extras) if extras else '(none)',
        'n':           int(fit['n']),
        'k':           int(fit['k']),
        'sd_w_iv':     sd_iv,
        'sd_w_out':    sd_out,
        'beta':        float(fit['params'][1]),
        'se':          float(fit['se'][1]),
        't':           float(fit['t'][1]),
        'p':           float(fit['p'][1]),
        'ci_lo':       float(fit['ci_lo'][1]),
        'ci_hi':       float(fit['ci_hi'][1]),
        'r2_model':    float(fit['r2']),
    }

## Step 4 — Run M1 (total) / M2 (+ total volume) / M3 (direct) for each exposure and outcome cell


In [ ]:
# Every rhythm metric through every outcome cell and specification.
results = []
SPEC = ([('Depression (PHQ-9 total)', 'PHQ9', c, lo, hi) for c, lo, hi in AGE_CELLS]
        + [('Cognitive function composite', 'COG_Z', '60+', 60, 999)])

for exp in EXPOSURES:
    for outcome_label, outcome_col, cell, lo, hi in SPEC:
        for model_label, extras in [
            ('M1 (Total, c-path)',        []),
            ('M2 (+ total volume)',       M2_EXTRA),
            ('M3 (Direct, c-prime path)', MED_VARS),
        ]:
            r = run_y_path_stdyx(exp, outcome_col, extras, age_min=lo, age_max=hi)
            r['outcome_label'] = f'{outcome_label}, {age_label(cell)}'
            r['model_label']   = model_label
            results.append(r)
            print(f"{exp:3s} {outcome_label[:24]:24s} {cell:6s} {model_label:26s} "
                  f"n={r['n']:,}  beta={r['beta']:+.3f} "
                  f"({r['ci_lo']:+.3f}, {r['ci_hi']:+.3f})  p={fmt_p(r['p'])}")
    print()

results_raw = pd.DataFrame(results)
results_raw.to_csv(R_RAW / 'script8_outcomes_full.csv', index=False)
print(f'\nWrote {R_RAW / "script8_outcomes_full.csv"}')


## Step 5 — Build Supplementary Table S15

One block per exposure and outcome cell; rows M1, M2 and M3 with n, STDYX beta (95% CI), p and R2.


In [ ]:
rows = []
for exp in EXPOSURES:
  for outcome_label in results_raw['outcome_label'].unique():
    block = results_raw[(results_raw['outcome_label'] == outcome_label)
                        & (results_raw['exposure'] == exp)].reset_index(drop=True)
    if not len(block):
        continue
    rows.append({'Outcome': f'{exp} -> {outcome_label}', 'Model': '', 'n': '',
                 'beta_STDYX (95% CI)': '', 'p': '', 'R2': ''})
    for _, r in block.iterrows():
        rows.append({
            'Outcome':             '',
            'Model':               r['model_label'],
            'n':                   f"{int(r['n']):,}",
            'beta_STDYX (95% CI)': fmt_ci(r['beta'], r['ci_lo'], r['ci_hi']),
            'p':                   fmt_p(r['p']),
            'R2':                  f"{r['r2_model']:.3f}",
        })

table3 = pd.DataFrame(rows)
print(table3.to_string(index=False))
table3.to_csv(R_SUPP / 'SuppTableS15_TotalDirect_WLS.csv', index=False)
try:
    with pd.ExcelWriter(R_SUPP / 'SuppTableS15_TotalDirect_WLS.xlsx', engine='openpyxl') as xw:
        table3.to_excel(xw, sheet_name='Supp Table S15', index=False)
        pd.DataFrame({'Notes': [
            'Survey-weighted multivariable linear regression (WTMEC4YR = WTMEC2YR / 2).',
            'Source: data/processed/8_nhanes_analytic.csv.',
            'Per-outcome analytic sample: IV + outcome + required predictors non-null (other variables not required, mirroring Mplus FIML).',
            'IV, outcome, and mediators (when in model) are z-standardised using SURVEY-WEIGHTED mean and SD on the per-outcome sample (Mplus convention under TYPE = COMPLEX + MLR). Covariates kept on raw scales.',
            'Reported beta = STDYX standardised metric -> outcome association (per 1 s.d. of the rhythm metric, in outcome s.d.).',
            'Standard errors: HC3 heteroscedasticity-consistent (MacKinnon-White) via wls_utils.wls_fit.',
            'M1 (Total, c-path): age, sex, body mass index, education, wear-time, sleep duration. No mediator.',
            'M2 (+ total volume): M1 + total daily MIMS volume.',
            'M3 (Direct, c′-path): M1 + KDM-BA advancement + log SII. The IV coefficient is the direct effect conditional on both mediators; matches Mplus c′ in Table 2.',
            'Total / direct decomposition under Baron-Kenny: c = c′ + a1*b1 + a2*b2.',
            'Cognition uses the simple composite Cog_composite_z (mean of memory / fluency / speed within-60+ z-scores) as the analogue of the Mplus latent COG factor.',
        ]}).to_excel(xw, sheet_name='Notes', index=False)
    print(f'Wrote {R_SUPP / "SuppTableS15_TotalDirect_WLS.xlsx"}')
except Exception as e:
    print(f'(xlsx skipped: {e})')

# Move superseded result/main/Table3_total_direct files, if present, to _superseded/.
for _stale in [R_MAIN / 'Table3_total_direct.csv', R_MAIN / 'Table3_total_direct.xlsx']:
    if _stale.exists():
        _dest = BASE / '_superseded' / _stale.name
        _dest.parent.mkdir(exist_ok=True)
        _stale.replace(_dest)
        print(f'superseded -> _superseded/{_stale.name}')


## Step 6 — Consistency check against Mplus total / direct coefficients

Compares the IV M1 and M3 STDYX coefficients with the Mplus STDYX values from
`M1_depression.out` / `M1_cognition.out` listed in `MPLUS_REF`.


In [ ]:
# Mplus STDYX reference values from M1_depression.out / M1_cognition.out (transcribed)
MPLUS_REF = {
    ('Depression (PHQ-9 total)', 'M1 (Total, c-path)'):       (+0.057, 'Total effect IV -> PHQ9'),
    ('Depression (PHQ-9 total)', 'M3 (Direct, c′-path)'):(+0.052, 'cp (Direct, c′-path) IV -> PHQ9'),
    ('Cognitive function composite (60+)', 'M1 (Total, c-path)'):       (-0.043, 'Total effect IV -> COG (latent)'),
    ('Cognitive function composite (60+)', 'M3 (Direct, c′-path)'):(-0.024, 'cp (Direct, c′-path) IV -> COG (latent)'),
}

print('Self-check (WLS-HC3 STDYX vs Mplus M1_*.out STDYX coefficients)')
print('-' * 100)
for (outcome_label, model_label), (mplus_beta, what) in MPLUS_REF.items():
    row = results_raw[(results_raw['outcome_label'] == outcome_label) &
                      (results_raw['model_label'] == model_label)]
    if row.empty:
        continue
    r = row.iloc[0]
    delta_beta = r['beta'] - mplus_beta
    flag = 'OK' if abs(delta_beta) < 0.005 else ('CLOSE' if abs(delta_beta) < 0.015 else 'CHECK')
    print(f'  {outcome_label[:36]:36s}  {model_label[:24]:24s}  '
          f'WLS={r["beta"]:+.3f}  Mplus={mplus_beta:+.3f}  '
          f'delta={delta_beta:+.3f} [{flag}]   ({what})')